 A small anatomical pipeline

Start from the Styx book's [anatomical preprocessing example](https://niwrap.dev/styxbook/examples/anat_preproc.html) (reorient, crop, BET, FAST), then add N4 bias correction, registration to MNI152NLin2009cAsym, the segmentation in template space, and a loop over subjects.

Work through the `TODO`s in order. Autocomplete and hover show the parameters. `03_pipeline_solution.ipynb` has a solution.

In [ ]:
import logging
import os
import shutil
from pathlib import Path

import niwrap
from nilearn import plotting
from niwrap import ants, fsl

BIDS = Path("../data/ds000001")
TEMPLATE = Path("../data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz")
SPACE = "MNI152NLin2009cAsym"
OUT = Path("../derivatives/workshop")

RUNNER = "auto"  # change to "dry" if Docker doesn't work on your machine
if RUNNER == "dry":
    runner = niwrap.use_dry()
else:
    runner = niwrap.use_auto(
        data_dir="../work/03",
        environ={"ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS": str(os.cpu_count())},
    )
    logging.getLogger(runner.logger_name).setLevel(logging.INFO)

`environ` is passed into every container. Without `ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS`, ANTs registration uses one core.

# Step 0: the book's example

The book's `anatomical_preprocessing`, updated for NiWrap 1.1: `fslreorient2std`, `robustfov` and `fast` need named outputs, or their output attributes are `None`. BET's `robust_iters` (`-R`) is dropped because it fails in the default FSL image; `robustfov` already removes the neck.

In [ ]:
def anatomical_preprocessing(t1w: Path) -> tuple[fsl.BetOutputs, fsl.FastOutputs]:
    reorient = fsl.fslreorient2std(input_image=t1w, output_image="t1w_reoriented.nii.gz")
    crop = fsl.robustfov(input_file=reorient.output_image, output_image="t1w_cropped.nii.gz")
    brain = fsl.bet(infile=crop.output_roi_volume, fractional_intensity=0.5, binary_mask=True)
    seg = fsl.fast(in_files=[brain.outfile], img_type=1, out_basename="t1w")
    return brain, seg


brain, seg = anatomical_preprocessing(BIDS / "sub-01/anat/sub-01_T1w.nii.gz")
seg.tissue_class_map

In [ ]:
if RUNNER != "dry":
    plotting.plot_roi(seg.tissue_class_map, bg_img=brain.outfile, title="sub-01 FAST")

# Step 1: bias-field correction

Run ANTs N4 between BET and FAST. Its output name is built with a helper, `ants.n4_bias_field_correction_corrected_output_params(...)`.

In [ ]:
def anatomical_preprocessing(t1w: Path) -> tuple[fsl.BetOutputs, Path, fsl.FastOutputs]:
    reorient = fsl.fslreorient2std(input_image=t1w, output_image="t1w_reoriented.nii.gz")
    crop = fsl.robustfov(input_file=reorient.output_image, output_image="t1w_cropped.nii.gz")
    brain = fsl.bet(infile=crop.output_roi_volume, fractional_intensity=0.5, binary_mask=True)
    # TODO 1: bias-correct brain.outfile with ants.n4_bias_field_correction
    #   (pass brain.binary_mask as mask_image), run FAST on the corrected image,
    #   and return the corrected image instead of brain.outfile.
    seg = fsl.fast(in_files=[brain.outfile], img_type=1, out_basename="t1w")
    return brain, brain.outfile, seg

# Step 2: register to the template

`ants.ants_registration_sy_n_quick_sh` runs `antsRegistrationSyNQuick.sh` (rigid, affine, SyN). Outputs include the warped image, the affine (`output_transform`) and the warp (`output_warp`).

In [ ]:
def register(t1w_brain: Path) -> ants.AntsRegistrationSyNQuickShOutputs:
    # TODO 2: register t1w_brain (moving) to TEMPLATE (fixed) in 3D
    #   with ants.ants_registration_sy_n_quick_sh and return its outputs.
    raise NotImplementedError("TODO 2")

# Step 3: warp the segmentation

Use `ants.ants_apply_transforms` with `GenericLabel` interpolation so labels stay integers. ANTs applies transforms last to first: list the warp before the affine.

In [ ]:
def warp_labels(labels: Path, xfm: ants.AntsRegistrationSyNQuickShOutputs) -> Path:
    # TODO 3: warp labels into TEMPLATE space with ants.ants_apply_transforms.
    #   Transforms are built with ants.ants_apply_transforms_transform_file_name_params,
    #   the output name with ants.ants_apply_transforms_warped_output_params.
    raise NotImplementedError("TODO 3")

# Step 4: one subject, start to finish

`process_subject` chains the steps and returns the files to keep. `save` copies them into `derivatives/` with BIDS names.

In [ ]:
def process_subject(t1w: Path) -> dict[str, Path]:
    brain, t1w_brain, seg = anatomical_preprocessing(t1w)
    xfm = register(t1w_brain)
    return {
        "desc-preproc_T1w.nii.gz": t1w_brain,
        "desc-brain_mask.nii.gz": brain.binary_mask,
        "dseg.nii.gz": seg.tissue_class_map,
        f"space-{SPACE}_desc-preproc_T1w.nii.gz": xfm.output_warped_image,
        # TODO 4: add the segmentation in template space (use warp_labels)
        #   and the two transform files from xfm.
    }


def save(outputs: dict[str, Path], subject: str) -> dict[str, Path]:
    out_dir = OUT / subject / "anat"
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = {}
    for suffix, src in outputs.items():
        saved[suffix] = out_dir / f"{subject}_{suffix}"
        if RUNNER != "dry":
            shutil.copy2(src, saved[suffix])
    return saved

In [ ]:
sub01 = save(process_subject(BIDS / "sub-01/anat/sub-01_T1w.nii.gz"), "sub-01")
sub01

In [ ]:
if RUNNER != "dry":
    plotting.plot_roi(sub01[f"space-{SPACE}_dseg.nii.gz"], bg_img=TEMPLATE, title="sub-01 tissue classes, MNI space")

# Step 5: every subject

A loop over subjects. sub-01 is already done.

In [ ]:
results = {"sub-01": sub01}
# TODO 5: loop over every sub-*/anat/sub-*_T1w.nii.gz under BIDS and, for each
#   subject not in results yet, run process_subject, save the outputs, and store
#   them in results[subject] ("sub-02", ...).

In [ ]:
if RUNNER != "dry":
    for subject, files in results.items():
        plotting.plot_anat(files[f"space-{SPACE}_desc-preproc_T1w.nii.gz"], title=subject, display_mode="z", cut_coords=5)

# Next: more subjects

For more than a few subjects, a workflow engine runs them in parallel, re-runs only what's missing, and submits to a cluster. Next section.